<h1><img src="../../../icons/tk_full_logo.svg" width="80" /> Fine-Tune GPT-OSS for Catalan Reasoning</h1>

Fine-tune GPT-OSS 20B with **Unsloth QLoRA** on Catalan chain-of-thought reasoning.

**Prerequisites:**
- Run `01-prepare-catalan-data.ipynb` first
- Model `unsloth/gpt-oss-20b-bnb-4bit` mirrored to MLflow
- GPU with 16GB+ VRAM

---
## 1. Setup

In [1]:
import os
import json
import torch
from pathlib import Path

from check_jupyter_flavor import check_flavor
check_flavor('fine-tuning')

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

max_seq_length = 4096  # Chain-of-thought can be long

✓ Running in correct environment: fine-tuning
  Fine-Tuning Lab (Unsloth, QLoRA, PEFT, TRL + ml-gpu)
CUDA: True
GPU: NVIDIA GB10
Memory: 128.5 GB


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


---
## 2. Load Training Data

In [2]:
from datasets import Dataset

DATA_PATH = Path("./data/catalan_reasoning_train.jsonl")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Run 01-prepare-catalan-data.ipynb first! Expected: {DATA_PATH}")

# Load JSONL format
training_data = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        training_data.append(json.loads(line))

print(f"Loaded: {len(training_data)} examples from {DATA_PATH}")

# Show sample structure
print("\nSample structure:")
sample = training_data[0]
print(f"  Keys: {list(sample.keys())}")
print(f"  Messages: {len(sample['messages'])} messages")
print(f"  Language: {sample.get('language', 'N/A')}")

# Show message roles
for msg in sample['messages']:
    thinking = ' (+thinking)' if 'thinking' in msg else ''
    print(f"    - {msg['role']}{thinking}")

Loaded: 200 examples from data/catalan_reasoning_train.jsonl

Sample structure:
  Keys: ['messages', 'language', 'original_language']
  Messages: 3 messages
  Language: Catalan
    - system
    - user
    - assistant (+thinking)


In [3]:
# Preview an example
print("="*60)
print("SAMPLE TRAINING EXAMPLE")
print("="*60)

sample = training_data[0]
for msg in sample['messages']:
    role = msg['role'].upper()
    content = msg['content'][:200] + '...' if len(msg['content']) > 200 else msg['content']
    print(f"\n[{role}]")
    print(content)
    if 'thinking' in msg:
        thinking = msg['thinking'][:200] + '...' if len(msg['thinking']) > 200 else msg['thinking']
        print(f"\n[{role} - THINKING]")
        print(thinking)

SAMPLE TRAINING EXAMPLE

[SYSTEM]
reasoning language: Catalan

You are an AI that formats its responses in simple, easy to understand language for children

[USER]
I'd like to pla a trip to Rome for 7 days. I want to see the main attractions like the Colosseum, Vatican City, and the Sistine Chapel, but I also want to explori some lesser-known sites. I'm a foodie...

[ASSISTANT]
**Rome 7-Day Itinerary: Història, Alimentació i Relaxació** Hi little explorer! Let's plan a fun and tasty trip to Rome where you'll see cool places, eat amazing food, and take breaks to relax. Heus a...

[ASSISTANT - THINKING]
Perfecte, a veure. L'usuari vol un viatge de 7 dies a Roma amb un equilibri entre les atraccions principals, alguns llocs ocults, bon menjar i moments de relaxació. Va esmentar el Coliseu, el Vaticà, ...


---
## 3. Load Model

In [4]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


def get_model_path(model_id: str) -> Path:
    """Get model path from MLflow."""
    model_name = model_id.replace('/', '-')
    mlflow_url = os.environ.get('MLFLOW_TRACKING_URI')
    token_url = os.environ.get('MLFLOW_KEYCLOAK_TOKEN_URL')
    
    token = None
    if token_url:
        resp = requests.post(token_url, data={
            'grant_type': 'password',
            'client_id': os.environ.get('MLFLOW_KEYCLOAK_CLIENT_ID', 'mlflow'),
            'client_secret': os.environ.get('MLFLOW_CLIENT_SECRET'),
            'username': os.environ.get('MLFLOW_AUTH_USERNAME'),
            'password': os.environ.get('MLFLOW_AUTH_PASSWORD'),
            'scope': 'openid'
        }, verify=False, timeout=30)
        resp.raise_for_status()
        token = resp.json()['access_token']
    
    headers = {'Authorization': f'Bearer {token}'} if token else {}
    
    resp = requests.get(
        f"{mlflow_url}/api/2.0/mlflow/model-versions/search",
        params={'filter': f"name='{model_name}'"},
        headers=headers, verify=False, timeout=30
    )
    resp.raise_for_status()
    
    versions = resp.json().get('model_versions', [])
    if not versions:
        raise ValueError(f"Model '{model_name}' not found in MLflow")
    
    latest = max(versions, key=lambda v: int(v['version']))
    run_id = latest['run_id']
    
    resp = requests.get(
        f"{mlflow_url}/api/2.0/mlflow/runs/get",
        params={'run_id': run_id},
        headers=headers, verify=False, timeout=30
    )
    resp.raise_for_status()
    experiment_id = resp.json()['run']['info']['experiment_id']
    
    for base in [Path('/home/jovyan/thinkube/mlflow'), Path.home() / 'thinkube' / 'mlflow']:
        path = base / 'artifacts' / experiment_id / run_id / 'artifacts' / 'model'
        if path.exists():
            return path
    
    raise FileNotFoundError("Model not found")


MODEL_ID = "unsloth/gpt-oss-20b-bnb-4bit"
model_path = get_model_path(MODEL_ID)
print(f"Model path: {model_path}")

Model path: /home/jovyan/thinkube/mlflow/artifacts/1/e0d35d5fb9eb4f5a8d4a26119a0dc6d4/artifacts/model


In [5]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(model_path),
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
    trust_remote_code=True,
    device_map={"": 0},
)

print(f"Loaded: {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"GPU Memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not import trl.trainer.alignprop_trainer: Failed to import trl.trainer.alignprop_trainer because of the following error (look up to see its traceback):
cannot import name 'DDPOStableDiffusionPipeline' from 'trl.models' (/usr/local/lib/python3.12/dist-packages/trl/models/__init__.py)


[unsloth_zoo.log|WARNING]Unsloth: Failed to import trl openenv: No module named 'trl.experimental'


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.4: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 119.697 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gpt_Oss does not support SDPA - switching to fast eager.


Loading checkpoint shards:   0%|          | 0/9 [00:00<?, ?it/s]

Loaded: 20,596,252,224 parameters
GPU Memory: 40.9 GB


---
## 4. Format Data for Training

Convert Harmony format with `thinking` field to GPT-OSS tokenized format.

GPT-OSS Harmony format for chain-of-thought:
- Thinking goes in `<|channel|>commentary` 
- Final answer goes in `<|channel|>final`

In [6]:
def format_for_harmony_training(example):
    """Convert Harmony JSON to GPT-OSS training format.
    
    The training data has:
    - system message with reasoning language instruction
    - user message with query
    - assistant message with 'thinking' (chain-of-thought) and 'content' (final)
    
    We need to format as proper harmony tokens:
    - Thinking: <|start|>assistant<|channel|>commentary<|message|>...<|end|>
    - Final: <|start|>assistant<|channel|>final<|message|>...<|end|>
    """
    messages = example["messages"]
    
    # Find messages by role
    system_msg = next((m for m in messages if m["role"] == "system"), None)
    user_msg = next(m for m in messages if m["role"] == "user")
    asst_msg = next(m for m in messages if m["role"] == "assistant")
    
    # Build the prompt part (system + user)
    prompt_messages = []
    if system_msg:
        prompt_messages.append({"role": "system", "content": system_msg["content"]})
    prompt_messages.append({"role": "user", "content": user_msg["content"]})
    
    prompt = tokenizer.apply_chat_template(
        prompt_messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # Build the assistant response in proper harmony format
    thinking = asst_msg.get("thinking", "")
    final_answer = asst_msg.get("content", "")
    
    # Chain-of-thought format:
    # 1. Commentary channel for thinking
    # 2. Final channel for answer
    assistant_response = ""
    
    if thinking:
        assistant_response += f"<|start|>assistant<|channel|>commentary<|message|>{thinking}<|end|>"
    
    if final_answer:
        assistant_response += f"<|start|>assistant<|channel|>final<|message|>{final_answer}<|end|>"
    
    # Remove the trailing generation prompt marker from tokenizer output
    if prompt.endswith("<|start|>assistant<|message|>"):
        prompt = prompt[:-len("<|start|>assistant<|message|>")]
    elif prompt.endswith("<|start|>assistant"):
        prompt = prompt[:-len("<|start|>assistant")]
    
    full_text = prompt + assistant_response
    
    return {"text": full_text}


# Convert all examples
formatted_data = [format_for_harmony_training(ex) for ex in training_data]
train_dataset = Dataset.from_list(formatted_data)
train_dataset = train_dataset.filter(lambda x: len(x["text"]) > 0)

print(f"Formatted: {len(train_dataset)} examples")
print(f"\nSample (showing harmony structure):")
sample = train_dataset[0]["text"]

# Show the assistant part clearly
if "<|start|>assistant" in sample:
    asst_start = sample.index("<|start|>assistant")
    print(f"...{sample[asst_start:asst_start+500]}...")

Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

Formatted: 200 examples

Sample (showing harmony structure):
...<|start|>assistant<|channel|>commentary<|message|>Perfecte, a veure. L'usuari vol un viatge de 7 dies a Roma amb un equilibri entre les atraccions principals, alguns llocs ocults, bon menjar i moments de relaxació. Va esmentar el Coliseu, el Vaticà, la Capella Sixtina i vol explorar llocs menys coneguts. A més, és un amant del menjar, així que he d'incloure restaurants i mercats imperdibles. També, alguns moments de descans. Primer, haig d'estructurar els dies de manera que no se sentin massa es...


---
## 5. Add LoRA

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} ({100*trainable/total:.3f}%)")

Unsloth: Making `model.base_model.model.model` require gradients
Trainable: 3,981,312 (0.019%)


---
## 6. Train

In [8]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# Calculate steps based on dataset size
# With ~250 examples, batch_size=1, grad_accum=4 -> ~60 steps per epoch
# Train for ~3 epochs
num_examples = len(train_dataset)
effective_batch_size = 4  # batch_size * grad_accum
steps_per_epoch = num_examples // effective_batch_size
num_epochs = 3
max_steps = steps_per_epoch * num_epochs

print(f"Training config:")
print(f"  Examples: {num_examples}")
print(f"  Steps per epoch: {steps_per_epoch}")
print(f"  Total steps: {max_steps}")

sft_config = SFTConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=max_steps,
    learning_rate=2e-4,
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.001,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
    report_to="none",
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=sft_config,
)

# Train on responses only - match the harmony format for chain-of-thought
# The response starts at <|start|>assistant<|channel|>commentary (for thinking)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start|>user<|message|>",
    response_part="<|start|>assistant<|channel|>commentary"
)

print("\nTraining...")
stats = trainer.train()
print(f"\nDone! Loss: {stats.training_loss:.4f}")

Training config:
  Examples: 200
  Steps per epoch: 50
  Total steps: 150


Unsloth: Tokenizing ["text"] (num_proc=24):   0%|          | 0/200 [00:00<?, ? examples/s]

Map (num_proc=24):   0%|          | 0/200 [00:00<?, ? examples/s]


Training...
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,3.380900
20,1.580600
30,1.421600
40,1.473100
50,1.345000
60,1.311800
70,1.345800
80,1.266500
90,1.334500
100,1.265200



Done! Loss: 1.4618


---
## 7. Test

In [10]:
FastLanguageModel.for_inference(model)

def parse_harmony_cot_response(text: str) -> dict:
    """Parse harmony format chain-of-thought response."""
    import re
    result = {"thinking": None, "final": None, "raw": text}
    
    # Extract commentary (thinking)
    commentary_match = re.search(
        r'<\|channel\|>commentary<\|message\|>(.*?)(?:<\|end\|>|<\|start\|>|$)', 
        text, 
        re.DOTALL
    )
    if commentary_match:
        result["thinking"] = commentary_match.group(1).strip()
    
    # Extract final answer
    final_match = re.search(
        r'<\|channel\|>final<\|message\|>(.*?)(?:<\|end\|>|$)', 
        text, 
        re.DOTALL
    )
    if final_match:
        result["final"] = final_match.group(1).strip()
    
    return result


# Test prompts in Catalan
test_prompts = [
    "Quant fa 15 × 23?",
    "Si tinc 8 pomes i en dono 3, quantes me'n queden?",
    "Quin és el resultat de (5 + 3) × 2?",
    "Un tren viatja a 60 km/h. Quant trigarà a recórrer 180 km?",
]

print("Testing Catalan reasoning:\n")

for prompt in test_prompts:
    system_msg = """reasoning language: Catalan

You are a helpful assistant that thinks through problems step by step.
Always show your reasoning process before giving the final answer."""
    
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt}
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(response_tokens, skip_special_tokens=False)
    
    parsed = parse_harmony_cot_response(response)
    
    print(f"[PREGUNTA] {prompt}")
    if parsed["thinking"]:
        thinking_preview = parsed["thinking"]
        print(f"[PENSAMENT] {thinking_preview}")
    if parsed["final"]:
        print(f"[RESPOSTA] {parsed['final']}")
    else:
        print(f"[RAW] {response}")
    print("-" * 60)

Testing Catalan reasoning:

[PREGUNTA] Quant fa 15 × 23?
[PENSAMENT] Bé, l'usuari m'ha demanat calcular 15 × 23. Anem a desglossar-ho pas a pas. Primer, recordo que 15 és 10 + 5 i 23 és 20 + 3. Per tant, puc usar la propietat distributiva per multiplicar cada part per separat. Això és, multipliquem 15 per 20 i després 15 per 3, i finalment sumem els resultats
[RESPOSTA] The result of \(15 \times 23\) is **345**.
------------------------------------------------------------
[PREGUNTA] Si tinc 8 pomes i en dono 3, quantes me'n queden?
[PENSAMENT] Bé, l'usuari té 8 pomes i en dona 3. Necessito trobar quantes li queden. Per resoldre-ho, restaré la quantitat donada de la quantitat inicial. Així que, 8 - 3 = 5. Llavors, li queden 5 pomes. Però, veig que l'usuari va escriure en català. He de respondre en català també. Així que, la resposta ha de ser "Li queden 5 pomes". No hi ha cap complicació addicional en aquest problema. És una simple resta.
[RESPOSTA] Li queden 5 pomes.
------------------

---
## 8. Save

In [11]:
OUTPUT_DIR = Path("/home/jovyan/thinkube/models/gpt-oss-catalan-reasoning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# LoRA adapters
LORA_DIR = OUTPUT_DIR / "lora"
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"LoRA adapters: {LORA_DIR}")

# Save training metadata
metadata = {
    "base_model": MODEL_ID,
    "task": "Catalan mathematical reasoning",
    "training_examples": len(train_dataset),
    "training_loss": stats.training_loss,
    "lora_r": 8,
    "lora_alpha": 16,
    "max_seq_length": max_seq_length,
    "source_dataset": "HuggingFaceH4/Multilingual-Thinking (Spanish → Catalan)",
    "translator": "projecte-aina/aina-translator-es-ca",
}

with open(OUTPUT_DIR / "training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata: {OUTPUT_DIR / 'training_metadata.json'}")

LoRA adapters: /home/jovyan/thinkube/models/gpt-oss-catalan-reasoning/lora
Metadata: /home/jovyan/thinkube/models/gpt-oss-catalan-reasoning/training_metadata.json


In [1]:
# Optional: Merge adapters into full model (takes more disk space but faster inference)
MERGE = True  # Set to True to merge

if MERGE:
    MERGED_DIR = OUTPUT_DIR / "merged"
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
    print(f"Merged model: {MERGED_DIR}")
else:
    print("Skipping merge (set MERGE=True to create merged model)")

NameError: name 'OUTPUT_DIR' is not defined

---
## Done!

The fine-tuned model is saved and ready for evaluation.

**Next step:** Run `03-evaluate-catalan-math.ipynb` to evaluate on ALIA-math-test benchmark.

### Output Files

| File | Description |
|------|-------------|
| `lora/` | LoRA adapters (small, can be merged with base model) |
| `training_metadata.json` | Training configuration and metrics |
| `merged/` | Full merged model (if MERGE=True) |